# ¿Hasta qué punto podemos confiar en AlphaFold?
**Bioinformática · Grado en Ciencia e Ingeniería de Datos · ULPGC**  
**Caso de estudio** · Mioglobina humana y subunidad beta de la hemoglobina humana

## 1. Exploración e hipótesis

| Proteína | UniProt | Organismo | Longitud canónica | Función y característica relevante |
|---|---|---|---:|---|
| Mioglobina | [P02144](https://www.uniprot.org/uniprotkb/P02144/entry) | *Homo sapiens* | 154 | Almacena oxígeno y facilita su difusión en el músculo. Es una globina monomérica con grupo hemo. |
| Hemoglobina beta | [P68871](https://www.uniprot.org/uniprotkb/P68871/entry) | *Homo sapiens* | 147 | Participa en el transporte de oxígeno por la sangre. En la hemoglobina adulta HbA, dos subunidades beta se ensamblan con dos alfa. |

**Hipótesis.** Los dos modelos tendrán perfiles de confianza predominantemente altos. La hipótesis se considerará apoyada si en ambos más del 90 % de los residuos tiene pLDDT superior a 90 y la diferencia entre esos porcentajes es de como máximo cinco puntos porcentuales. También se localizarán las caídas puntuales para saber qué información pierde la media.

Los visores de [mioglobina](https://alphafold.ebi.ac.uk/entry/P02144) y [hemoglobina beta](https://alphafold.ebi.ac.uk/entry/P68871) muestran la estructura coloreada por pLDDT y el mapa PAE. Las coordenadas del visor se contrastan con el perfil por residuo y con la representación de los carbonos alfa que se genera más abajo.

## 2. Obtención y control de los datos

Ejecutar de arriba abajo en Colab o Jupyter con `numpy`, `pandas` y `matplotlib` instalados. Por defecto se consulta la API y se usan las URLs del mismo registro. Si falla la descarga, desde las fichas enlazadas arriba se pueden guardar el PDB y el JSON del PAE en la carpeta del notebook, con los nombres originales. Si falla la consulta de metadatos, vuelve a ejecutar la celda cuando la conexión esté disponible.


In [ ]:
from datetime import date
from pathlib import Path
from urllib.request import urlopen
from urllib.error import URLError
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

NOMBRES = {"P02144": "Mioglobina", "P68871": "Hemoglobina beta"}
URL_AF = "https://alphafold.ebi.ac.uk"
CARPETA_LOCAL = Path(".")


Se descarga la secuencia canónica en formato FASTA desde UniProt. Después se consulta la API de AlphaFold DB y se selecciona el modelo que cubre esa secuencia completa. El identificador, la versión y los enlaces del PDB y del PAE se leen del mismo registro.


In [ ]:
def solicitar(url):
    with urlopen(url, timeout=15) as respuesta:
        if respuesta.status != 200:
            raise ValueError(f"HTTP {respuesta.status} para {url}")
        return respuesta.read().decode("utf-8")


def seleccionar_modelo(acceso, registros, secuencia):
    candidatos = [
        registro for registro in registros
        if registro.get("uniprotAccession") == acceso
        and registro.get("sequence") == secuencia
        and registro.get("sequenceStart") == 1
        and registro.get("sequenceEnd") == len(secuencia)
        and registro.get("isComplex") is False
        and registro.get("pdbUrl") and registro.get("paeDocUrl")
    ]
    if len(candidatos) != 1:
        raise ValueError(f"{acceso}: se esperaba un único modelo monomérico de secuencia canónica completa.")
    return candidatos[0]


modelos = {}
procedencias = []
for acceso in NOMBRES:
    url_fasta = f"https://rest.uniprot.org/uniprotkb/{acceso}.fasta"
    url_api = f"{URL_AF}/api/prediction/{acceso}"
    try:
        fasta = solicitar(url_fasta).splitlines()
        registros = json.loads(solicitar(url_api))
    except (URLError, TimeoutError) as error:
        raise RuntimeError(f"No se pudieron consultar los datos de {acceso}: {error}") from error
    if not fasta or not fasta[0].startswith(f">sp|{acceso}|"):
        raise ValueError(f"El FASTA descargado no corresponde a la entrada revisada {acceso}.")
    secuencia = "".join(fasta[1:]).strip()
    modelo = seleccionar_modelo(acceso, registros, secuencia)
    modelos[acceso] = modelo
    procedencias.append({
        "UniProt": acceso, "Modelo": modelo["entryId"], "Versión": modelo.get("latestVersion"),
        "Fecha de consulta": date.today().isoformat(), "FASTA": url_fasta,
        "API AlphaFold DB": url_api, "PDB": modelo["pdbUrl"],
        "PAE": modelo["paeDocUrl"], "pLDDT medio (API)": modelo["globalMetricValue"],
        "% muy alta (API, redondeado)": 100 * modelo["fractionPlddtVeryHigh"],
        "% alta (API, redondeado)": 100 * modelo["fractionPlddtConfident"],
        "% baja (API, redondeado)": 100 * modelo["fractionPlddtLow"],
        "% muy baja (API, redondeado)": 100 * modelo["fractionPlddtVeryLow"],
    })
display(pd.DataFrame(procedencias).set_index("UniProt"))


En una línea `ATOM` del PDB se identifican el átomo, el residuo, la cadena y las coordenadas `x, y, z`. AlphaFold DB utiliza el campo B para almacenar el pLDDT por residuo y repite el valor en sus átomos. Leer únicamente `CA` da un valor por residuo y evita ponderar más los aminoácidos con más átomos. Aquí se leen las columnas de anchura fija del formato PDB.


In [ ]:
RESIDUOS = dict(zip(
    "ALA CYS ASP GLU PHE GLY HIS ILE LYS LEU MET ASN PRO GLN ARG SER THR VAL TRP TYR".split(),
    "ACDEFGHIKLMNPQRSTVWY",
))


def leer_ca(pdb, secuencia):
    filas = []
    for linea in pdb.splitlines():
        if not linea.startswith("ATOM  ") or linea[12:16].strip() != "CA":
            continue
        if linea[16] != " " or linea[26] != " ":
            raise ValueError("Se encontraron posiciones alternativas o códigos de inserción.")
        filas.append({
            "posición": int(linea[22:26]), "residuo": linea[17:20].strip(),
            "cadena": linea[21], "x": float(linea[30:38]), "y": float(linea[38:46]),
            "z": float(linea[46:54]), "pLDDT": float(linea[60:66]),
        })
    tabla = pd.DataFrame(filas, columns=["posición", "residuo", "cadena", "x", "y", "z", "pLDDT"])
    if tabla["cadena"].nunique() != 1 or tabla["posición"].duplicated().any():
        raise ValueError("El PDB debe tener una única cadena y posiciones sin duplicados.")
    if tabla["posición"].tolist() != list(range(1, len(secuencia) + 1)):
        raise ValueError("La numeración del PDB no cubre la secuencia canónica en orden.")
    if "".join(RESIDUOS.get(aa, "?") for aa in tabla["residuo"]) != secuencia:
        raise ValueError("Los residuos del PDB no coinciden con la secuencia canónica.")
    if not tabla["pLDDT"].between(0, 100).all():
        raise ValueError("Se encontraron valores de pLDDT fuera de [0, 100].")
    if not np.isfinite(tabla[["x", "y", "z"]].to_numpy()).all():
        raise ValueError("Se encontraron coordenadas no finitas.")
    return tabla


def leer_pae(contenido, longitud):
    matriz = np.asarray(json.loads(contenido)[0]["predicted_aligned_error"], dtype=float)
    if matriz.shape != (longitud, longitud):
        raise ValueError(f"Dimensión PAE {matriz.shape}. Se esperaba {(longitud, longitud)}.")
    if not np.isfinite(matriz).all() or (matriz < 0).any():
        raise ValueError("El PAE debe contener valores finitos y no negativos.")
    return matriz


def obtener_archivo(url):
    nombre = url.rsplit("/", 1)[-1]
    ruta = CARPETA_LOCAL / nombre
    if ruta.is_file():
        return ruta.read_text(encoding="utf-8"), f"Archivo local {nombre}"
    try:
        return solicitar(url), "Descarga directa"
    except (URLError, TimeoutError) as error:
        raise RuntimeError(
            f"Descarga no disponible para {nombre}: {error}. "
            f"Guarda el archivo de la ficha de AlphaFold con su nombre original en {CARPETA_LOCAL.resolve()}."
        ) from error


## 3. Análisis del pLDDT

Se aplica el umbral de la práctica de forma estricta. Un valor de 70 corta un tramo. Las cuatro bandas del resumen son muy baja (<50), baja ([50, 70)), alta ([70, 90]) y muy alta (>90).


In [ ]:
def detectar_tramos(posiciones, valores, umbral=70, minimo=3):
    tramos = []
    inicio = anterior = None
    for pos, valor in zip(posiciones, valores, strict=True):
        consecutivo = anterior is not None and pos == anterior + 1
        if inicio is not None and (not consecutivo or valor >= umbral):
            if anterior - inicio + 1 >= minimo:
                tramos.append((inicio, anterior))
            inicio = None
        if valor < umbral and inicio is None:
            inicio = pos
        anterior = pos
    if inicio is not None and anterior - inicio + 1 >= minimo:
        tramos.append((inicio, anterior))
    return tramos


# Un salto corta el primer tramo y el último se guarda al terminar.
assert detectar_tramos([1, 2, 3, 5, 6, 7, 8, 9, 10, 11],
                       [55, 65, 66, 60, 62, 70, 69, 67, 60, 58]) == [(1, 3), (8, 11)]
# El 70 exacto separa dos rachas de longitud insuficiente.
assert detectar_tramos([1, 2, 3, 4, 5], [65, 66, 70, 67, 68]) == []
print("Pruebas del detector superadas.")


In [ ]:
tablas, matrices, fuentes_archivos = {}, {}, []
for acceso, modelo in modelos.items():
    print(f"Descargando o leyendo {acceso}")
    pdb, origen_pdb = obtener_archivo(modelo["pdbUrl"])
    pae, origen_pae = obtener_archivo(modelo["paeDocUrl"])
    print("Ejemplos de líneas ATOM:")
    print("\n".join([linea for linea in pdb.splitlines() if linea.startswith("ATOM  ")][:3]))
    tablas[acceso] = leer_ca(pdb, modelo["sequence"])
    matrices[acceso] = leer_pae(pae, len(tablas[acceso]))
    fuentes_archivos.append({"UniProt": acceso, "Origen PDB": origen_pdb,
                             "Origen PAE": origen_pae, "Lectura": date.today().isoformat()})
    display(tablas[acceso].head())
display(pd.DataFrame(fuentes_archivos))


In [ ]:
resumen, tramos_por_proteina = {}, {}
for acceso, tabla in tablas.items():
    p = tabla["pLDDT"]
    tramos = detectar_tramos(tabla["posición"], p)
    tramos_por_proteina[acceso] = tramos
    resumen[acceso] = {
        "Proteína": NOMBRES[acceso], "N": len(p), "Media": p.mean(), "Mediana": p.median(),
        "% muy baja": 100 * (p < 50).mean(), "% baja": 100 * ((p >= 50) & (p < 70)).mean(),
        "% alta": 100 * p.between(70, 90).mean(), "% muy alta": 100 * (p > 90).mean(),
        "Tramos <70": tramos, "Tramo más largo": max(tramos, key=lambda t: t[1] - t[0], default=None),
    }
    print(NOMBRES[acceso], "· posiciones con pLDDT < 70:",
          tabla.loc[p < 70, "posición"].tolist())
display(pd.DataFrame.from_dict(resumen, orient="index").round(2))


In [ ]:
BANDAS = [(0, 50, "#ff7d45"), (50, 70, "#ffdb13"), (70, 90, "#65cbf3"), (90, 100, "#0053d6")]
fig = plt.figure(figsize=(12, 7), layout="constrained")
for fila, (acceso, tabla) in enumerate(tablas.items()):
    ax = fig.add_subplot(2, 2, 2 * fila + 1)
    for inferior, superior, color in BANDAS:
        ax.axhspan(inferior, superior, color=color, alpha=.15)
    for umbral in (50, 70, 90):
        ax.axhline(umbral, color="#626262", lw=.7, ls="--")
    ax.plot(tabla["posición"], tabla["pLDDT"], color="#15263b", lw=1.5)
    bajos = tabla[tabla["pLDDT"] < 70]
    ax.scatter(bajos["posición"], bajos["pLDDT"], color="#ad490b", zorder=3)
    ax.set(title=NOMBRES[acceso], xlabel="Posición del residuo (UniProt)", ylabel="pLDDT",
           ylim=(0, 103), xlim=(1, max(map(len, tablas.values()))))

    vista = fig.add_subplot(2, 2, 2 * fila + 2, projection="3d")
    p = tabla["pLDDT"]
    colores = np.select([p < 50, p < 70, p <= 90], [b[2] for b in BANDAS[:3]], default=BANDAS[3][2])
    vista.plot(tabla["x"], tabla["y"], tabla["z"], color="#a3acb7", lw=1)
    vista.scatter(tabla["x"], tabla["y"], tabla["z"], c=colores, s=14, depthshade=False)
    for r in bajos.itertuples(index=False, name=None):
        vista.text(r[3], r[4], r[5], str(r[0]), color="#834520")
    vista.set_box_aspect(np.ptp(tabla[["x", "y", "z"]].to_numpy(), axis=0))
    vista.set_title("Traza Cα del modelo, coloreada por pLDDT")
    vista.set_axis_off()
plt.show()


La traza Cα vuelve a dibujar las coordenadas de la estructura publicada y aplica el mismo esquema de bandas que el visor. Así se puede localizar en 3D el número de residuo que aparece en el perfil. La escala 0–100 permite comparar perfiles y los porcentajes compensan que las longitudes sean distintas. Los índices por sí solos no establecen homología entre posiciones. Para identificar residuos equivalentes se requeriría un alineamiento de secuencia o estructura.

## 4. PAE y análisis de dos regiones de la hemoglobina beta

La celda de fila `i` y columna `j` representa el error esperado en la posición del residuo `i` al alinear usando el residuo `j`. El mapa conserva la dirección de la matriz original. Un valor bajo expresa mayor confianza en esa posición relativa.

En la hemoglobina beta se seleccionan las **hélices 59–75 (A) y 82–95 (B)**, anotadas en UniProt. Ambas contienen histidinas relacionadas con el entorno del hemo, la distal en la posición 64 y la proximal en la 93. Estos tramos pertenecen a una misma cadena. Los rótulos A y B se usan aquí para identificar las dos regiones del análisis, no como nombres de hélices en la nomenclatura clásica.

[Anotaciones de UniProt P68871](https://www.uniprot.org/uniprotkb/P68871/entry#structure) · [Guía de PAE de EMBL-EBI](https://www.ebi.ac.uk/training/online/courses/alphafold/inputs-and-outputs/evaluating-alphafolds-predicted-structures-using-confidence-scores/pae-a-measure-of-global-confidence-in-alphafold-predictions/)


In [ ]:
from matplotlib.patches import Rectangle

inicio_a, fin_a = 59, 75
inicio_b, fin_b = 82, 95
maximo_pae = max(matriz.max() for matriz in matrices.values())
fig, ejes = plt.subplots(1, 2, figsize=(11, 4.5), layout="constrained")
for ax, (acceso, matriz) in zip(ejes, matrices.items(), strict=True):
    n = len(matriz)
    imagen = ax.imshow(matriz, cmap="viridis", vmin=0, vmax=maximo_pae, origin="upper",
                       extent=(.5, n + .5, n + .5, .5), interpolation="nearest")
    ax.set(title=NOMBRES[acceso], xlabel="Residuo de alineamiento (j)",
           ylabel="Residuo evaluado (i)")
    if acceso == "P68871":
        for inicio, fin in [(inicio_a, fin_a), (inicio_b, fin_b)]:
            ancho = fin - inicio + 1
            ax.add_patch(Rectangle((inicio - .5, inicio - .5), ancho, ancho,
                                   fill=False, edgecolor="white", linewidth=1.2))
fig.colorbar(imagen, ax=ejes, label="PAE (Å)")
plt.show()

hb = matrices["P68871"]
a = slice(inicio_a - 1, fin_a)
b = slice(inicio_b - 1, fin_b)
medias_pae = pd.DataFrame(
    [[hb[a, a].mean(), hb[a, b].mean()], [hb[b, a].mean(), hb[b, b].mean()]],
    index=pd.Index(["A · 59–75", "B · 82–95"], name="Región evaluada"),
    columns=pd.Index(["A · 59–75", "B · 82–95"], name="Región de alineamiento"),
)
local = tablas["P68871"]["pLDDT"]
media_a, media_b = local.iloc[a].mean(), local.iloc[b].mean()
display(medias_pae.round(2))
display(pd.Series({"A · 59–75": media_a, "B · 82–95": media_b}, name="pLDDT medio").round(2))


## 5. Conclusión

La conclusión toma las cifras de las celdas anteriores. Las medias internas del PAE incluyen la diagonal. Las cifras muestran la diferencia relativa de incertidumbre entre bloques, sin asumir un umbral universal de PAE.


In [ ]:
mb, hbb = resumen["P02144"], resumen["P68871"]
diferencia = abs(mb["% muy alta"] - hbb["% muy alta"])
apoya = min(mb["% muy alta"], hbb["% muy alta"]) > 90 and diferencia <= 5
resultado = "apoyan" if apoya else "no apoyan"
frase_tramos = (
    f"El detector devolvió {len(mb['Tramos <70'])} tramos en mioglobina y "
    f"{len(hbb['Tramos <70'])} en hemoglobina beta."
)
aa, ab = medias_pae.iloc[0]
ba, bb = medias_pae.iloc[1]
if min(ab, ba) > max(aa, bb):
    interpretacion = "En ambas direcciones, el error medio entre regiones supera el error medio dentro de cada región."
elif max(ab, ba) <= min(aa, bb):
    interpretacion = "El error medio entre regiones no supera los valores internos en ninguna dirección."
else:
    interpretacion = "La comparación del error entre regiones con el error interno depende de la dirección o de la región interna elegida."
patron = (
    "Las dos regiones tienen confianza local alta y la colocación relativa resulta más incierta que su estructura interna."
    if min(media_a, media_b) >= 70 and min(ab, ba) > max(aa, bb)
    else "Este par de regiones no muestra simultáneamente confianza local alta en ambas y un PAE entre ellas superior a los dos valores internos en ambas direcciones."
)
conclusion = f'''
Los datos {resultado} la hipótesis de perfiles predominantemente altos. El porcentaje de residuos con
pLDDT superior a 90 fue de {mb['% muy alta']:.2f} % en mioglobina y {hbb['% muy alta']:.2f} % en hemoglobina beta,
con una diferencia de {diferencia:.2f} puntos porcentuales. Las medias de pLDDT fueron {mb['Media']:.2f} y
{hbb['Media']:.2f}, con medianas de {mb['Mediana']:.2f} y {hbb['Mediana']:.2f}, respectivamente. La media
resume la tendencia general, mientras que el perfil y la traza tridimensional permiten localizar valores
puntuales inferiores a 70. {frase_tramos}

En hemoglobina beta, el pLDDT medio de las regiones 59–75 y 82–95 fue de {media_a:.2f} y {media_b:.2f}.
El PAE medio dentro de ambas regiones fue de {aa:.2f} Å y {bb:.2f} Å. Al evaluar la primera región con
alineamiento en la segunda se obtuvo {ab:.2f} Å y en sentido inverso, {ba:.2f} Å.
{interpretacion} {patron}

La entrada analizada de hemoglobina beta representa una cadena aislada. Sus puntuaciones describen esa
predicción y dejan abierta la disposición de las cadenas en el tetrámero funcional, sus interfaces y
la geometría de unión del grupo hemo. La mioglobina sirve como referencia para comparar perfiles de
confianza. Para medir el error de las coordenadas se necesitaría una estructura experimental adecuada y
una correspondencia explícita entre residuos. El pLDDT y el PAE son estimaciones internas de AlphaFold y
dependen de la representación estructural evaluada.
'''
display(Markdown(conclusion.strip()))
